# Notebook 2 — Dataset Construction

Builds supervised rolling-window datasets for all selected experiment configurations. Each output contains chronological train, validation, and test splits, binary top-20% labels, portfolio/date metadata, and realized next-period returns for backtesting.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
DATA_DIR = Path("../data")
DATASET_DIR = DATA_DIR / "datasets"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

DATA = {
    "monthly_portfolios": DATA_DIR / "25_Portfolios_5x5_Usable_Monthly.csv",
    "daily_portfolios": DATA_DIR / "25_Portfolios_5x5_Usable_Daily.csv",
    "monthly_ff3": DATA_DIR / "F-F-3_Research_Usable_Monthly.csv",
    "daily_ff3": DATA_DIR / "F-F-3_Research_Usable_Daily.csv",
    "monthly_ff5": DATA_DIR / "F-F-5_Research_Usable_Monthly.csv",
    "daily_ff5": DATA_DIR / "F-F-5_Research_Usable_Daily.csv",
}

for name, path in DATA.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path.resolve()}")
print("All cleaned input files found.")

All cleaned input files found.


In [3]:
RUN_EXPERIMENTS = [
    "monthly_ff3",
    "monthly_ff5",
    "daily_ff3",
    "daily_ff5",
]

EXPERIMENT_CONFIG = {
    "monthly_ff3": {"frequency": "monthly", "factor_model": "ff3", "window": 12, "periods_per_year": 12},
    "monthly_ff5": {"frequency": "monthly", "factor_model": "ff5", "window": 12, "periods_per_year": 12},
    "daily_ff3": {"frequency": "daily", "factor_model": "ff3", "window": 60, "periods_per_year": 252},
    "daily_ff5": {"frequency": "daily", "factor_model": "ff5", "window": 60, "periods_per_year": 252},
}

unknown = set(RUN_EXPERIMENTS) - set(EXPERIMENT_CONFIG)
if unknown:
    raise ValueError(f"Unknown experiments: {sorted(unknown)}")

In [4]:
def load_dataset(path, frequency):
    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.str.strip()
    if df.columns[0] != "Date":
        df = df.rename(columns={df.columns[0]: "Date"})
    date_text = df["Date"].astype(str).str.strip()
    fmt = "%Y%m" if frequency == "monthly" else "%Y%m%d"
    df["Date"] = pd.to_datetime(date_text, format=fmt, errors="raise")
    for col in df.columns[1:]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df.sort_values("Date").drop_duplicates("Date").reset_index(drop=True)

monthly_returns = load_dataset(DATA["monthly_portfolios"], "monthly")
daily_returns = load_dataset(DATA["daily_portfolios"], "daily")
monthly_ff3 = load_dataset(DATA["monthly_ff3"], "monthly")
daily_ff3 = load_dataset(DATA["daily_ff3"], "daily")
monthly_ff5 = load_dataset(DATA["monthly_ff5"], "monthly")
daily_ff5 = load_dataset(DATA["daily_ff5"], "daily")

In [5]:
def merge_returns_and_factors(returns_df, factors_df):
    merged = returns_df.merge(factors_df, on="Date", how="inner", validate="one_to_one")
    portfolio_cols = [c for c in returns_df.columns if c != "Date"]
    factor_cols = [c for c in factors_df.columns if c != "Date"]
    return merged, portfolio_cols, factor_cols

def create_top_labels(returns_df, top_pct=0.20):
    portfolio_cols = [c for c in returns_df.columns if c != "Date"]
    future_returns = returns_df[portfolio_cols].shift(-1)
    ranks = future_returns.rank(axis=1, ascending=False, method="first")
    top_n = max(1, int(np.ceil(len(portfolio_cols) * top_pct)))
    labels = (ranks <= top_n).astype("Int8")
    labels.insert(0, "Date", returns_df["Date"])
    return labels

def align_labels_to_merged(merged_df, labels_df):
    return merged_df[["Date"]].merge(labels_df, on="Date", how="left", validate="one_to_one")

In [6]:
monthly_labels = create_top_labels(monthly_returns)
daily_labels = create_top_labels(daily_returns)

sources = {
    "monthly_ff3": (monthly_returns, monthly_ff3, monthly_labels),
    "monthly_ff5": (monthly_returns, monthly_ff5, monthly_labels),
    "daily_ff3": (daily_returns, daily_ff3, daily_labels),
    "daily_ff5": (daily_returns, daily_ff5, daily_labels),
}

prepared = {}
for experiment in RUN_EXPERIMENTS:
    returns_df, factors_df, labels_df = sources[experiment]
    merged, portfolio_cols, factor_cols = merge_returns_and_factors(returns_df, factors_df)
    aligned_labels = align_labels_to_merged(merged, labels_df)
    prepared[experiment] = {
        "merged": merged,
        "labels": aligned_labels,
        "portfolio_cols": portfolio_cols,
        "factor_cols": factor_cols,
    }
    print(experiment, merged.shape, merged["Date"].min(), "->", merged["Date"].max())

monthly_ff3 (1198, 30) 1926-07-01 00:00:00 -> 2026-04-01 00:00:00
monthly_ff5 (754, 32) 1963-07-01 00:00:00 -> 2026-04-01 00:00:00
daily_ff3 (26252, 30) 1926-07-01 00:00:00 -> 2026-05-28 00:00:00
daily_ff5 (15832, 32) 1963-07-01 00:00:00 -> 2026-05-28 00:00:00


In [7]:
def build_rolling_windows(merged_df, labels_df, window_size, portfolio_cols, factor_cols):
    X, y, metadata = [], [], []
    merged_df = merged_df.sort_values("Date").reset_index(drop=True)
    labels_df = labels_df.sort_values("Date").reset_index(drop=True)

    for i in range(window_size, len(merged_df) - 1):
        prediction_date = merged_df.loc[i, "Date"]
        realization_date = merged_df.loc[i + 1, "Date"]
        for portfolio in portfolio_cols:
            window = merged_df.loc[i-window_size:i-1, [portfolio] + factor_cols].to_numpy(dtype=np.float32)
            label = labels_df.loc[i, portfolio]
            realized_next_return = merged_df.loc[i + 1, portfolio]
            if np.isnan(window).any() or pd.isna(label) or pd.isna(realized_next_return):
                continue
            X.append(window)
            y.append(int(label))
            metadata.append({
                "Date": prediction_date,
                "RealizationDate": realization_date,
                "Portfolio": portfolio,
                "RealizedNextReturn": float(realized_next_return),
            })
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.int8), pd.DataFrame(metadata)

def chronological_date_split(X, y, meta, train_fraction=0.70, validation_fraction=0.15):
    unique_dates = np.array(sorted(meta["Date"].unique()))
    train_end = int(len(unique_dates) * train_fraction)
    val_end = int(len(unique_dates) * (train_fraction + validation_fraction))
    date_sets = {
        "train": unique_dates[:train_end],
        "validation": unique_dates[train_end:val_end],
        "test": unique_dates[val_end:],
    }
    splits = {}
    for split_name, dates in date_sets.items():
        mask = meta["Date"].isin(dates).to_numpy()
        splits[split_name] = {
            "X": X[mask], "y": y[mask],
            "meta": meta.loc[mask].reset_index(drop=True),
        }
    return splits

In [9]:
all_splits = {}
for experiment in RUN_EXPERIMENTS:
    cfg = EXPERIMENT_CONFIG[experiment]
    p = prepared[experiment]
    X, y, meta = build_rolling_windows(
        p["merged"], p["labels"], cfg["window"], p["portfolio_cols"], p["factor_cols"]
    )
    splits = chronological_date_split(X, y, meta)
    all_splits[experiment] = splits
    print(f"{experiment}: X={X.shape}, y={y.shape}, positive rate={y.mean():.4f}")
    for split_name, d in splits.items():
        print(split_name, d["X"].shape, d["meta"]["Date"].min(), "->", d["meta"]["Date"].max())

monthly_ff3: X=(29625, 12, 5), y=(29625,), positive rate=0.2000
train (20725, 12, 5) 1927-07-01 00:00:00 -> 1996-07-01 00:00:00
validation (4450, 12, 5) 1996-08-01 00:00:00 -> 2011-05-01 00:00:00
test (4450, 12, 5) 2011-06-01 00:00:00 -> 2026-03-01 00:00:00
monthly_ff5: X=(18525, 12, 7), y=(18525,), positive rate=0.2000
train (12950, 12, 7) 1964-07-01 00:00:00 -> 2007-08-01 00:00:00
validation (2775, 12, 7) 2007-09-01 00:00:00 -> 2016-11-01 00:00:00
test (2800, 12, 7) 2016-12-01 00:00:00 -> 2026-03-01 00:00:00
daily_ff3: X=(654775, 60, 5), y=(654775,), positive rate=0.2000
train (458325, 60, 5) 1926-09-14 00:00:00 -> 1995-03-06 00:00:00
validation (98225, 60, 5) 1995-03-07 00:00:00 -> 2010-10-11 00:00:00
test (98225, 60, 5) 2010-10-12 00:00:00 -> 2026-05-27 00:00:00
daily_ff5: X=(394275, 60, 7), y=(394275,), positive rate=0.2000
train (275975, 60, 7) 1963-09-25 00:00:00 -> 2007-08-03 00:00:00
validation (59150, 60, 7) 2007-08-06 00:00:00 -> 2016-12-23 00:00:00
test (59150, 60, 7) 2016-

In [10]:
for experiment, splits in all_splits.items():
    out_dir = DATASET_DIR / experiment
    out_dir.mkdir(parents=True, exist_ok=True)
    for split_name, d in splits.items():
        np.save(out_dir / f"X_{split_name}.npy", d["X"])
        np.save(out_dir / f"y_{split_name}.npy", d["y"])
        d["meta"].to_csv(out_dir / f"meta_{split_name}.csv", index=False)
    print("Saved", experiment, "to", out_dir.resolve())

Saved monthly_ff3 to C:\Users\kyler\Documents\VS_Code\Finance Code\ML using FAMA and FRENCH\data\datasets\monthly_ff3
Saved monthly_ff5 to C:\Users\kyler\Documents\VS_Code\Finance Code\ML using FAMA and FRENCH\data\datasets\monthly_ff5
Saved daily_ff3 to C:\Users\kyler\Documents\VS_Code\Finance Code\ML using FAMA and FRENCH\data\datasets\daily_ff3
Saved daily_ff5 to C:\Users\kyler\Documents\VS_Code\Finance Code\ML using FAMA and FRENCH\data\datasets\daily_ff5


In [11]:
for experiment in RUN_EXPERIMENTS:
    out_dir = DATASET_DIR / experiment
    X = np.load(out_dir / "X_train.npy")
    y = np.load(out_dir / "y_train.npy")
    meta = pd.read_csv(out_dir / "meta_train.csv")
    assert len(X) == len(y) == len(meta)
    assert {"Date", "RealizationDate", "Portfolio", "RealizedNextReturn"}.issubset(meta.columns)
    print(experiment, X.shape, y.shape, meta.shape)

monthly_ff3 (20725, 12, 5) (20725,) (20725, 4)
monthly_ff5 (12950, 12, 7) (12950,) (12950, 4)
daily_ff3 (458325, 60, 5) (458325,) (458325, 4)
daily_ff5 (275975, 60, 7) (275975,) (275975, 4)
